# Lab 5 — Notebook 03: From Annotations to a Custom Webcam Detector

**Core duration: 65 minutes**, plus 20 minutes of team planning. Training time depends on hardware; the instructor should benchmark it before class.

**Route:** frames/capture 8 min → annotation/splits 10 min → validate and display labels 15 min → short training 15 min → reload/evaluate/infer 12 min → webcam hand-off 5 min.

Use the existing `data/`, `examples/`, and `scripts/` directories. No variables from other notebooks are required. Student TODO cells are independent of later worked examples.

**Two dataset modes:**

- **Toy workflow check (included):** six synthetic shape images, four training and two validation. This can demonstrate loading labels, saving weights, and reloading them. It cannot establish useful detector performance. A short run may predict no objects.
- **Real dataset (instructor/team supplied):** switch `DATASET_ROOT` below to a prepared two- or three-class dataset. Use independent collection sessions for its partitions. This is the recommended classroom route for a meaningful custom model.

The repo does **not** include a real custom dataset or a prepared custom checkpoint. The configuration below makes these explicit rather than assuming they exist. Training is controlled by `RUN_TRAINING`; skipped training is not a completed training exercise.


## 1. Inspect a frame and collect a few images (8 min)

Start with the local fallback video. It is a synthetic pan over one coffee photograph, useful for testing video decoding—not a source of independent training and test examples.

OpenCV reads BGR; Matplotlib displays RGB. Release the capture even when reading fails. Use the existing desktop script for continuous camera access rather than an endless notebook loop.


In [ ]:
from pathlib import Path
# Support notebooks started in lab05/ or in the repository root.
LAB_DIR = Path.cwd().resolve()
if (LAB_DIR / "lab05" / "data" / "digits.csv").is_file():
    LAB_DIR = LAB_DIR / "lab05"
if not (LAB_DIR / "data" / "digits.csv").is_file():
    raise FileNotFoundError("Run this notebook in lab05/ or the repository root; keep data/ and scripts/ in place.")

import sys, subprocess, importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

video_path = LAB_DIR / "data/sample_video.mp4"
frame = None
if importlib.util.find_spec("cv2") is None:
    print("VIDEO SKIPPED — install the Lab 5 dependencies before the camera exercise.")
else:
    import cv2
    cap = cv2.VideoCapture(str(video_path))
    try:
        ok, frame = cap.read()
        if not ok:
            raise RuntimeError("Could not decode the fallback video. Check the path and codec support.")
    finally:
        cap.release()
    plt.figure(figsize=(7, 4))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Fallback video frame")
    plt.show()

# Copy the printed command into a local terminal. Paths use this notebook's interpreter.
capture_command = [sys.executable, str(LAB_DIR / "scripts/webcam_capture.py"),
                   "--source", "0", "--output", str(LAB_DIR / "captures")]
if sys.platform == "win32":
    print(subprocess.list2cmdline(capture_command))
else:
    import shlex
    print(shlex.join(capture_command))


**Core action:** Run the capture command in a local terminal. SPACE saves a frame; Q exits. Capture a few permitted images with different backgrounds, distances, and orientations. If no camera is available, use instructor images for annotation and continue the supplied toy workflow.

**Question:** Why would randomly splitting neighboring frames from one video give an overly optimistic evaluation?

**Your observation:** TODO.


## 2. Annotate and define independent splits (10 min)

Use [CVAT Online](https://app.cvat.ai/) or an instructor-provided export. Accounts and access should be arranged before class. Core practice is annotating **two or three images**; larger collection belongs to project work.

1. Create a standalone task; no project is required for this exercise.
2. Define a small, consistent class vocabulary and annotate every visible instance of those classes with rectangles.
3. Agree how to label partial objects. Keep background-only images, with no object annotations.
4. Save and export **Ultralytics YOLO Detection**. Keep original images locally. Inspect the export: do not assume it already contains the required splits.
5. Match image and label stems and check the class-ID order against `data.yaml`.

Use only images you have permission to upload; avoid capturing identifiable bystanders. Export availability can depend on your account. An instructor export is a valid fallback.

**Project splitting rule:** Keep a recording session together. Collect training, validation, and final test sessions separately; preserve enough examples of each class. Apply augmentation only to training. Use validation to choose settings and reserve the final test until choices are fixed.

The audit below catches exact duplicate image bytes across train/validation, but cannot prove independence or detect all near-duplicates. Record session provenance separately.

**Classroom folder contract:** Reorganize an export into `images/train/`, `images/val/`, `labels/train/`, `labels/val/`, plus `data.yaml` with `names`. Preserve originals. For this notebook use flat folders; YOLO also supports other layouts, but this helper intentionally validates only this simple one. Use empty `.txt` files for confirmed background images so a missing label cannot silently hide an export error.

[CVAT format documentation](https://docs.cvat.ai/docs/dataset_management/formats/format-yolo-ultralytics/) explains export variants.


## 3. Validate and display the dataset (15 min)

One Detection annotation row is:
```text
class_id x_center y_center width height
```
Box coordinates are normalized by image width/height. For pixel bounds:
$$x_c=(x_{min}+x_{max})/(2W),\quad y_c=(y_{min}+y_{max})/(2H),$$
$$w=(x_{max}-x_{min})/W,\quad h=(y_{max}-y_{min})/H.$$

Read the checks, run them, and visually inspect several images. Syntactically valid boxes can still be incorrect. The helper checks both center/size ranges **and box edges**, class IDs, readable images, label pairs, and exact train/validation duplicates.

It writes a **new local configuration with an absolute dataset path**, avoiding the ambiguous `path: .` in exported YAML files. It does not modify the supplied YAML or split your data automatically.

**The audit function is supplied infrastructure:** run it and interpret its report; you are not expected to implement or memorize the helper in this lab. Focus on the class vocabulary, normalized coordinates, split counts, and visual checks.


In [ ]:
import yaml
import hashlib
from collections import Counter

DATASET_ROOT = (LAB_DIR / "examples/yolo_dataset").resolve()
# For a real prepared dataset, change the line above, e.g. LAB_DIR / "project_dataset".
SOURCE_YAML = DATASET_ROOT / "data.yaml"
TOY_MODE = DATASET_ROOT == (LAB_DIR / "examples/yolo_dataset").resolve()
config = yaml.safe_load(SOURCE_YAML.read_text())
if not isinstance(config, dict):
    raise ValueError("data.yaml must contain a mapping with names.")
names = config.get("names")
if isinstance(names, list):
    names = dict(enumerate(names))
if not isinstance(names, dict) or any(type(k) is not int for k in names):
    raise ValueError("names must be a list or a dictionary with integer keys.")
if sorted(names) != list(range(len(names))) or not names:
    raise ValueError("Class IDs must be contiguous from zero.")
if any(not isinstance(v, str) or not v.strip() for v in names.values()):
    raise ValueError("Class names must be nonempty strings.")
if len(set(names.values())) != len(names):
    raise ValueError("Class names must be distinct.")
print("Classes:", names)
print("TOY WORKFLOW CHECK — metrics are not meaningful" if TOY_MODE else "REAL DATASET — verify session independence manually")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
def read_labels(path, class_names):
    boxes = []
    for line_no, line in enumerate(path.read_text().splitlines(), 1):
        if not line.strip():
            continue
        fields = line.split()
        if len(fields) != 5:
            raise ValueError(f"{path.name}:{line_no}: need 5 Detection fields, not pose/segmentation/track labels.")
        values = np.asarray([float(x) for x in fields])
        cls, xc, yc, bw, bh = values
        if not np.isfinite(values).all() or cls != int(cls) or int(cls) not in class_names:
            raise ValueError(f"Invalid class or non-finite value in {path}:{line_no}")
        if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
            raise ValueError(f"Invalid normalized box in {path}:{line_no}")
        eps = 1e-6  # tolerate rounding at image boundaries
        if xc-bw/2 < -eps or yc-bh/2 < -eps or xc+bw/2 > 1+eps or yc+bh/2 > 1+eps:
            raise ValueError(f"Box extends beyond image in {path}:{line_no}")
        boxes.append((int(cls), xc, yc, bw, bh))
    return boxes

def audit_split(dataset_root, split, class_names):
    image_dir = dataset_root / "images" / split
    label_dir = dataset_root / "labels" / split
    if not image_dir.is_dir() or not label_dir.is_dir():
        raise ValueError(f"Missing images/{split} or labels/{split} directory.")
    images = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
    if not images:
        raise ValueError(f"No images in {image_dir}")
    stems = [p.stem for p in images]
    if len(stems) != len(set(stems)):
        raise ValueError("Duplicate image stems make label pairing ambiguous.")
    orphan_labels = {p.stem for p in label_dir.glob("*.txt")} - set(stems)
    if orphan_labels:
        raise ValueError(f"Labels without images: {sorted(orphan_labels)}")
    counts, hashes, labels_by_image = Counter(), set(), {}
    background = 0
    for image_path in images:
        with Image.open(image_path) as image:
            image.load()
            if image.width <= 0 or image.height <= 0:
                raise ValueError(f"Invalid image dimensions: {image_path}")
        label_path = label_dir / (image_path.stem + ".txt")
        if not label_path.is_file():
            raise ValueError(f"Missing label: {label_path}. Confirm background images and supply empty labels.")
        boxes = read_labels(label_path, class_names)
        labels_by_image[image_path] = boxes
        counts.update(box[0] for box in boxes)
        background += int(len(boxes) == 0)
        hashes.add(hashlib.sha256(image_path.read_bytes()).hexdigest())
    return {"images": images, "boxes": labels_by_image, "counts": counts,
            "hashes": hashes, "background": background}

audit = {split: audit_split(DATASET_ROOT, split, names) for split in ["train", "val"]}
if audit["train"]["hashes"] & audit["val"]["hashes"]:
    raise ValueError("Exact duplicate image bytes occur in both training and validation.")
for split, info in audit.items():
    print(split, "images:", len(info["images"]), "backgrounds:", info["background"],
          "objects:", {names[k]: info["counts"][k] for k in names})
    missing = [names[k] for k in names if info["counts"][k] == 0]
    if missing:
        raise ValueError(f"Classes absent from {split}: {missing}. Revise collection/splits before training.")

workflow_dir = LAB_DIR / "runs/lab05_workflow"
workflow_dir.mkdir(parents=True, exist_ok=True)
DATA_YAML = workflow_dir / "resolved_data.yaml"
DATA_YAML.write_text(yaml.safe_dump({"path": str(DATASET_ROOT),
    "train": "images/train", "val": "images/val", "names": names}, sort_keys=False))
print("Training configuration:", DATA_YAML)
print("Audit passed. Still inspect boxes and session independence manually.")


In [ ]:
import matplotlib.patches as patches
# Display two training images and one validation image; never inspect final test here.
examples = audit["train"]["images"][:2] + audit["val"]["images"][:1]
fig, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 4))
for ax, image_path in zip(np.atleast_1d(axes), examples):
    split = image_path.parent.name
    image = Image.open(image_path).convert("RGB")
    W, H = image.size
    ax.imshow(image)
    for cls, xc, yc, bw, bh in audit[split]["boxes"][image_path]:
        left, top = (xc-bw/2)*W, (yc-bh/2)*H
        ax.add_patch(patches.Rectangle((left, top), bw*W, bh*H, fill=False, edgecolor="lime", linewidth=2))
        ax.text(left, top, names[cls], va="bottom", bbox={"facecolor":"white", "alpha":0.8})
    ax.set_title(f"{split}: {image_path.name}")
    ax.axis("off")
plt.tight_layout()
plt.show()


**Core check:** Read one annotation line, explain its five values, and point to its box. Are all visible objects of the chosen classes labeled? Why does the duplicate-file check not protect against neighboring video frames?

**Your response:** TODO.


## 4. Run one short fine-tuning job (15 min budget)

Start from pretrained YOLO11n and adapt its weights to your dataset's classes. This differs from the previous notebook's **inference only** use. Ultralytics manages batches, loss, backpropagation, and optimizer updates.

Set `RUN_TRAINING=True` for the guided run. Defaults are CPU, small images, batch 4, and `workers=0` for simpler notebook behavior on Windows. Two epochs on the toy dataset test the workflow only; even a real dataset's short run is not a convergence guarantee. A GPU may use `DEVICE=0` if already configured.

The code uses a unique run name, then reads the actual training output directory. It does not assume that a previous `runs/detect/train/` is today's run. Rerunning starts a new job. If class time is short, use an instructor-prepared **matching custom checkpoint** in the next section and state that you did not train it yourself.


In [ ]:
# Set this once before class. Download only when explicitly enabled.
ALLOW_DOWNLOAD = False
weight_candidates = [LAB_DIR / "weights/yolo11n.pt", LAB_DIR / "yolo11n.pt", LAB_DIR.parent / "yolo11n.pt"]
WEIGHTS = next((p for p in weight_candidates if p.is_file()), weight_candidates[0])
MODEL_SOURCE = str(WEIGHTS) if WEIGHTS.is_file() else ("yolo11n.pt" if ALLOW_DOWNLOAD else None)

from datetime import datetime
RUN_TRAINING = False  # Set True to perform the guided training step.
DEVICE = "cpu"
EPOCHS = 2 if TOY_MODE else 5
IMGSZ = 320
best_path = None
run_dir = None
YOLO = None
try:
    from ultralytics import YOLO
except ImportError as exc:
    print("TRAINING UNAVAILABLE — install Lab 5 dependencies:", exc)

if not RUN_TRAINING:
    print("TRAINING NOT RUN. Set RUN_TRAINING=True and rerun this cell, or use a matching instructor checkpoint below.")
elif YOLO is None:
    print("TRAINING BLOCKED — missing dependencies.")
elif MODEL_SOURCE is None:
    print("TRAINING BLOCKED — provide local yolo11n.pt or explicitly enable download.")
else:
    detector = YOLO(MODEL_SOURCE)
    run_name = "guided_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    detector.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ,
                   device=DEVICE, batch=4, workers=0, seed=42,
                   project=str(workflow_dir), name=run_name, exist_ok=False,
                   cache=False, plots=True, amp=False)
    run_dir = Path(detector.trainer.save_dir)
    best_path = run_dir / "weights/best.pt"
    if not best_path.is_file():
        raise FileNotFoundError(f"Training did not produce {best_path}; inspect training logs.")
    print("Saved custom checkpoint:", best_path)
    print("Training log:", run_dir / "results.csv")


## 5. Reload, validate, and inspect predictions (12 min)

Use the saved `best.pt` or set `PREPARED_CHECKPOINT` to the instructor's local custom weights. A pretrained COCO model is **not** a substitute for a model trained on your custom labels.

Validation helped choose the best epoch, so these results are **validation results, not a final independent test**. The toy dataset's metrics cannot establish real-world performance. For your project, reserve a separately collected final test session and evaluate it only after design choices are fixed.

The notebook checks class names/order, but that cannot verify a checkpoint's training provenance. For a prepared checkpoint, confirm its dataset and collection sessions with the instructor.


In [ ]:
PREPARED_CHECKPOINT = None  # Example: LAB_DIR / "weights/classroom_custom_best.pt"
checkpoint = Path(PREPARED_CHECKPOINT).resolve() if PREPARED_CHECKPOINT is not None else best_path
custom_model = None
metrics = None
if checkpoint is None:
    print("VALIDATION NOT RUN — finish training or provide a matching custom checkpoint.")
elif not checkpoint.is_file():
    raise FileNotFoundError(f"Custom checkpoint not found: {checkpoint}")
elif YOLO is None:
    print("VALIDATION BLOCKED — install Lab 5 dependencies.")
else:
    custom_model = YOLO(str(checkpoint))
    model_names = custom_model.names
    if isinstance(model_names, list):
        model_names = dict(enumerate(model_names))
    if model_names != names:
        raise ValueError(f"Checkpoint vocabulary {model_names} does not match dataset vocabulary {names}.")
    metrics = custom_model.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ,
                               device=DEVICE, batch=4, workers=0,
                               project=str(workflow_dir), name="validation", plots=True)
    print(f"Validation mAP50:    {metrics.box.map50:.3f}")
    print(f"Validation mAP50–95: {metrics.box.map:.3f}")
    if TOY_MODE:
        print("TOY WORKFLOW CHECK ONLY — do not report these as project performance.")

if run_dir is not None and (run_dir / "results.csv").is_file():
    log = pd.read_csv(run_dir / "results.csv")
    log.columns = log.columns.str.strip()
    print(log.tail(3).to_string(index=False))
    loss_columns = [c for c in log if c.startswith("train/") and c.endswith("loss")]
    if loss_columns:
        ax = log.plot(x="epoch", y=loss_columns, figsize=(8, 4), title="Training losses — not test accuracy")
        ax.set_ylabel("Loss")
        plt.tight_layout()
        plt.show()


In [ ]:
if custom_model is None:
    print("CUSTOM INFERENCE NOT RUN — load a matching checkpoint above.")
else:
    sample_paths = audit["val"]["images"][:2]
    fig, axes = plt.subplots(1, len(sample_paths), figsize=(6 * len(sample_paths), 4))
    for ax, image_path in zip(np.atleast_1d(axes), sample_paths):
        prediction = custom_model.predict(str(image_path), conf=0.25, imgsz=IMGSZ,
                                          device=DEVICE, verbose=False)[0]
        ax.imshow(prediction.plot()[..., ::-1])
        ax.set_title(f"{image_path.name}: {len(prediction.boxes)} detections")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    print("Compare these predictions with the ground-truth annotations. Zero detections is a result to discuss.")


**Core interpretation:** Report the dataset mode, whether you trained or used a prepared checkpoint, the checkpoint path, and one success or failure. Explain why lower training loss does not guarantee correct live predictions.

**Your response:** TODO. If training/inference was unavailable, explicitly record the unfinished steps instead of inventing metrics.


## 6. Use the custom checkpoint in the webcam script (5 min)

The command below passes your saved weights to the **existing** `webcam_detection.py`. It prints commands rather than opening a camera inside the notebook. Run one in a local terminal; Q exits. The existing script runs inference on CPU.

For toy shapes, show printed/displayed shapes to the camera only as an exploratory domain-shift test; useful performance is not expected after a tiny run. The coffee fallback video does not contain the toy custom classes and is only a video-I/O check for that model. For real classes use matching objects or a suitable recorded clip.

Record one observed limitation and whether the live system responds quickly enough. Do not equate model-only timing with end-to-end webcam speed.


In [ ]:
if custom_model is None or checkpoint is None:
    print("CUSTOM WEBCAM HAND-OFF NOT READY — first reload a custom checkpoint.")
else:
    webcam_command = [sys.executable, str(LAB_DIR / "scripts/webcam_detection.py"),
                      "--model", str(checkpoint), "--source", "0", "--imgsz", str(IMGSZ), "--conf", "0.25"]
    video_command = webcam_command.copy()
    video_command[video_command.index("--source") + 1] = str(video_path)
    for label, command in [("Webcam", webcam_command), ("Fallback video (check class suitability)", video_command)]:
        print(label)
        if sys.platform == "win32":
            print(subprocess.list2cmdline(command))
        else:
            import shlex
            print(shlex.join(command))
    print("Core hand-off ready; run the command and record the observed behavior.")


## Team project checkpoint (20 min, separate from the 65-minute workflow)

Agree a small first milestone before collecting a large dataset. Complete this plan:

| Decision | Your team response |
|---|---|
| Webcam application and useful output | TODO |
| Two or three initial classes | TODO |
| Annotation rules, including partial objects and backgrounds | TODO |
| Training collection sessions and conditions | TODO |
| Separate validation and final test sessions | TODO |
| Baseline and measurable success criterion | TODO |
| Failure conditions to investigate | TODO |
| Available training hardware and fallback plan | TODO |
| First milestone: custom checkpoint running in the script | TODO |

**Before leaving:** Save your notebook, annotation/export files, dataset configuration, and checkpoint. Keep captures and training outputs out of Git; retain them in durable project storage. The existing `lab05/.gitignore` covers captures and runs.

**Success means:** You can explain and demonstrate the path from images + labels to custom weights and inference. A toy workflow run proves the plumbing, not a useful solution. Your real project must provide independent evaluation and honest limitations.

### Instructor preparation

For the recommended real-data route, supply a small annotated dataset with session-based partitions, confirm that its class names match any fallback checkpoint, pre-download YOLO11n weights, and time the short training run on classroom hardware. Reserve time to resolve setup before this lab. Do not assume the toy dataset or fallback coffee video can establish project quality.

References: [Ultralytics training](https://docs.ultralytics.com/modes/train/), [validation](https://docs.ultralytics.com/modes/val/), [detection dataset format](https://docs.ultralytics.com/datasets/detect/).
